In [ ]:
# 1. Install the downloader
!pip install yt-dlp

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import random
import glob
import yt_dlp

# --- CONFIGURATION ---
CSV_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/metadata_csv_files/'
OUTPUT_FOLDER = '/content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/'
MIN_LEN = 120  # 2 mins
MAX_LEN = 360  # 6 mins

if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

def get_grouped_data(filepath):
    df = pd.read_csv(filepath, sep=None, engine='python', skipinitialspace=True, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    grouped = df.groupby('youtube_id').agg(
        time_start=('time_start', 'min'),
        label=('label', 'first')
    ).reset_index()
    return grouped

def download_clip(video_id, start_time, label):
    """
    Returns True if clip was downloaded or already exists properly.
    Returns False if download failed for any reason.
    """
    duration = random.randint(MIN_LEN, MAX_LEN)
    end_time = start_time + duration

    clean_label = str(label).replace(" ", "_")
    output_name = f"{clean_label}_{video_id}.mp4"
    output_path = os.path.join(OUTPUT_FOLDER, output_name)

    # Check if already properly downloaded
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        if size_mb > 1:
            print(f"Already exists ({size_mb:.1f} MB), skipping: {output_name}")
            return True
        else:
            print(f"Found incomplete file ({size_mb:.1f} MB), re-downloading...")
            os.remove(output_path)

    print(f"\n--- Processing: {video_id} ---")
    print(f"Label: {label}")
    print(f"Clip: {start_time}s → {end_time}s ({duration}s = ~{duration//60}m{duration%60}s)")

    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
        'outtmpl': output_path,
        'external_downloader': 'ffmpeg',
        'external_downloader_args': {
            'ffmpeg_i': ['-ss', str(start_time)],
            'ffmpeg': ['-t', str(duration), '-c', 'copy']
        },
        'quiet': False,
        'no_warnings': True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([f"https://www.youtube.com/watch?v={video_id}"])

        # Verify the file actually exists and is substantial
        if os.path.exists(output_path) and os.path.getsize(output_path) / (1024 * 1024) > 1:
            size_mb = os.path.getsize(output_path) / (1024 * 1024)
            print(f"✓ Saved: {output_name} ({size_mb:.1f} MB)")
            return True
        else:
            print(f"✗ File missing or too small after download: {output_name}")
            if os.path.exists(output_path):
                os.remove(output_path)  # clean up bad file
            return False

    except Exception as e:
        print(f"✗ Failed {video_id}: {e}")
        if os.path.exists(output_path):
            os.remove(output_path)  # clean up partial file
        return False


def process_file(csv_path, test_mode=False):
    """Process one CSV file. In test_mode, downloads one successful video and stops."""
    csv_name = os.path.basename(csv_path)
    print(f"\n{'='*50}")
    print(f"Processing: {csv_name}")
    print(f"{'='*50}")

    df_grouped = get_grouped_data(csv_path)
    print(f"Unique videos in this file: {len(df_grouped)}")

    if test_mode:
        candidates = df_grouped.sample(frac=1).reset_index(drop=True)
        for _, row in candidates.iterrows():
            print(f"\nTrying: {row['youtube_id']}")
            success = download_clip(row['youtube_id'], row['time_start'], row['label'])
            if success:
                print(f"✓ Got one video from {csv_name}")
                return
        print(f"✗ No available videos found in {csv_name}")

    else:
        for _, row in df_grouped.iterrows():
            download_clip(row['youtube_id'], row['time_start'], row['label'])


# --- MAIN ---
RUN_TEST_ONLY = True

all_files = glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
print(f"Found {len(all_files)} CSV files: {[os.path.basename(f) for f in all_files]}\n")

for csv_path in all_files:
    process_file(csv_path, test_mode=RUN_TEST_ONLY)

print("\nAll done.")

Found 43 CSV files: ['with_smoke.csv', 'volcanic_eruption.csv', 'tropical_cyclone.csv', 'thunderstorm.csv', 'tornado.csv', 'train_accident.csv', 'under_construction.csv', 'on_fire.csv', 'wildfire.csv', 'truck_accident.csv', 'traffic_jam.csv', 'van_accident.csv', 'motorcycle_accident.csv', 'storm_surge.csv', 'ice_storm.csv', 'rockslide_rockfall.csv', 'heavy_rainfall.csv', 'nuclear_explosion.csv', 'sinkhole.csv', 'fog.csv', 'snowslide_avalanche.csv', 'landslide.csv', 'ship_accident.csv', 'snow_covered.csv', 'derecho.csv', 'oil_spil.csv', 'hailstorm.csv', 'bus_accident.csv', 'blocked.csv', 'dust_sand_storm.csv', 'fire_whirl.csv', 'collapsed.csv', 'car_accident.csv', 'damaged.csv', 'flooded.csv', 'drought.csv', 'dust_devil.csv', 'dirty_contamined.csv', 'bicycle_accident.csv', 'earthquake.csv', 'mudslide_mudflow.csv', 'burned.csv', 'airplane_accident.csv']


Processing: with_smoke.csv
Unique videos in this file: 78

Trying: 1Z2K6lDt76M

--- Processing: 1Z2K6lDt76M ---
Label: with_smoke
Clip

ERROR: [youtube] 76qnTsnUymI: Video unavailable


✗ Failed 76qnTsnUymI: ERROR: [youtube] 76qnTsnUymI: Video unavailable

Trying: Bkyod1P6LO0

--- Processing: Bkyod1P6LO0 ---
Label: tropical cyclone
Clip: 0s → 312s (312s = ~5m12s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=Bkyod1P6LO0
[youtube] Bkyod1P6LO0: Downloading webpage
[youtube] Bkyod1P6LO0: Downloading android vr player API JSON
[info] Bkyod1P6LO0: Downloading 1 format(s): 136+140
[download] Destination: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/tropical_cyclone_Bkyod1P6LO0.mp4
[download] 100% of   13.06MiB in 00:00:01 at 7.24MiB/s
✓ Saved: tropical_cyclone_Bkyod1P6LO0.mp4 (13.1 MB)
✓ Got one video from tropical_cyclone.csv

Processing: thunderstorm.csv
Unique videos in this file: 37

Trying: djRRJN5G20A

--- Processing: djRRJN5G20A ---
Label: thunderstorm
Clip: 0s → 148s (148s = ~2m28s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=djRRJN5G20A
[youtube] djRRJN5G20A: Downloading webpage
[youtube] djRRJN5G20A: Downl

ERROR: [youtube] djRRJN5G20A: Video unavailable


✗ Failed djRRJN5G20A: ERROR: [youtube] djRRJN5G20A: Video unavailable

Trying: fAEZa3PqkFk

--- Processing: fAEZa3PqkFk ---
Label: thunderstorm
Clip: 0s → 129s (129s = ~2m9s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=fAEZa3PqkFk
[youtube] fAEZa3PqkFk: Downloading webpage
[youtube] fAEZa3PqkFk: Downloading android vr player API JSON
[info] fAEZa3PqkFk: Downloading 1 format(s): 137+140
[download] Destination: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/thunderstorm_fAEZa3PqkFk.mp4
[download] 100% of   37.27MiB in 00:01:37 at 391.80KiB/s
✓ Saved: thunderstorm_fAEZa3PqkFk.mp4 (37.3 MB)
✓ Got one video from thunderstorm.csv

Processing: tornado.csv
Unique videos in this file: 24

Trying: Kj9vYIvRdmQ

--- Processing: Kj9vYIvRdmQ ---
Label: tornado
Clip: 34s → 372s (338s = ~5m38s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=Kj9vYIvRdmQ
[youtube] Kj9vYIvRdmQ: Downloading webpage
[youtube] Kj9vYIvRdmQ: Downloading android vr player

...

✗ Failed B6lWTy4WBHY: ERROR: [youtube] B6lWTy4WBHY: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

Trying: m2gwFkCU1Q4

--- Processing: m2gwFkCU1Q4 ---
Label: bicycle accident
Clip: 90s → 382s (292s = ~4m52s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=m2gwFkCU1Q4
[youtube] m2gwFkCU1Q4: Downloading webpage
[youtube] m2gwFkCU1Q4: Downloading android vr player API JSON
[info] m2gwFkCU1Q4: Downloading 1 format(s): 399+140
[download] Destination: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/bicycle_accident_m2gwFkCU1Q4.mp4
[download] 100% of   95.50MiB in 00:02:34 at 632.63KiB/s
✓ Saved: bicycle_accident_m2gwFkCU1

ERROR: [youtube] sCJ99bApo6o: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


✗ Failed sCJ99bApo6o: ERROR: [youtube] sCJ99bApo6o: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies

Trying: PjUJrWqSFNY

--- Processing: PjUJrWqSFNY ---
Label: airplane accident
Clip: 26s → 275s (249s = ~4m9s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=PjUJrWqSFNY
[youtube] PjUJrWqSFNY: Downloading webpage
[youtube] PjUJrWqSFNY: Downloading android vr player API JSON
[info] PjUJrWqSFNY: Downloading 1 format(s): 397+140
[download] Destination: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/airplane_accident_PjUJrWqSFNY.mp4
[download] 100% of  555.51KiB in 00:00:00 at 734.77KiB/s
✗ File missing or too small after 

ERROR: [youtube] 7g3NRWg7q9Q: Video unavailable


✗ Failed 7g3NRWg7q9Q: ERROR: [youtube] 7g3NRWg7q9Q: Video unavailable

Trying: kGg6lk4dTnA

--- Processing: kGg6lk4dTnA ---
Label: airplane accident
Clip: 18s → 338s (320s = ~5m20s)
[youtube] Extracting URL: https://www.youtube.com/watch?v=kGg6lk4dTnA
[youtube] kGg6lk4dTnA: Downloading webpage
[youtube] kGg6lk4dTnA: Downloading android vr player API JSON
[info] kGg6lk4dTnA: Downloading 1 format(s): 399+140
[download] Destination: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/actual_videos/airplane_accident_kGg6lk4dTnA.mp4
[download] 100% of   41.61MiB in 00:02:43 at 260.26KiB/s
✓ Saved: airplane_accident_kGg6lk4dTnA.mp4 (41.6 MB)
✓ Got one video from airplane_accident.csv

All done.


In [ ]:
import os
# Let's check the actual folder structure to find the correct path
base_path = '/content/drive/MyDrive/'
# Searching for the 'Project' folder or its parents to verify names
for root, dirs, files in os.walk(base_path):
    if 'Project' in dirs:
        print(f"Found Project folder at: {os.path.join(root, 'Project')}")
        # List subdirectories to find 'metadata_csv_files'
        project_path = os.path.join(root, 'Project')
        for p_root, p_dirs, p_files in os.walk(project_path):
             if 'metadata_csv_files' in p_dirs:
                 print(f"✓ SUCCESS! Found metadata folder at: {os.path.join(p_root, 'metadata_csv_files')}")
        break

Found Project folder at: /content/drive/MyDrive/4th year/S2/RL/Project
✓ SUCCESS! Found metadata folder at: /content/drive/MyDrive/4th year/S2/RL/Project/Dataset/Incidents/metadata_csv_files
